# 0. Setup

Load the modules

In [ ]:
from seal import *

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.models import Model
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from imgaug import augmenters as iaa
import tensorflow as tf
import random
from sklearn import svm
import requests
import tensorflow
from keras.models import Sequential
import warnings
from sklearn.preprocessing import normalize
warnings.filterwarnings(action='ignore')

%matplotlib inline

# 1. Key Generation
- Define the key path
- If there is no generated key pair, use this code to generate key set.

In [ ]:
PK_PATH = '/workspace/shared_data/keys/public.key'
SK_PATH = '/workspace/shared_data/keys/secret.key'  # client-only in a real deployment
GK_PATH = '/workspace/shared_data/keys/galois.key'
RL_PATH = '/workspace/shared_data/keys/relin.key'

# 2. CKKS scheme's parameter setting (In the SEAL-Python)

In [ ]:
parms = EncryptionParameters(scheme_type.ckks)
poly_modulus_degree = 16384
parms.set_poly_modulus_degree(poly_modulus_degree)
parms.set_coeff_modulus(CoeffModulus.Create(
        poly_modulus_degree, [60, 40, 40, 40, 60]))
scale = 2.0**40
context = SEALContext(parms)
ckks_encoder = CKKSEncoder(context)
slot_count = ckks_encoder.slot_count()
print(f'Number of slots: {slot_count}')

# --- AUTH: LOAD the enrolled keys. Do NOT regenerate - the query ciphertext must
#     be encrypted under the SAME public key as the enrolled ctxt1 / the server's
#     galois+relin keys, or the HE result decrypts to garbage. ---
public_key = PublicKey(); public_key.load(context, PK_PATH)
secret_key = SecretKey(); secret_key.load(context, SK_PATH)
galois_key = GaloisKeys(); galois_key.load(context, GK_PATH)
relin_keys = RelinKeys(); relin_keys.load(context, RL_PATH)
print('Keys loaded from /workspace/shared_data/keys')

encryptor = Encryptor(context, public_key)
evaluator = Evaluator(context)
decryptor = Decryptor(context, secret_key)

# 3. Load dataset
- Load the test dataset which is saved in the training

In [ ]:
TESTSET_PATH = '/workspace/shared_data/data/test_data.npy'
test_data = np.load(TESTSET_PATH, allow_pickle=True)
test_data.shape

# 4. Load models

- Load the models which are saved in the training

In [ ]:
from keras import backend as K
def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

swish = tensorflow.keras.activations.swish

custom_objects={"f1_m": f1_m, "precision_m":precision_m, "recall_m":recall_m}

def sigmoid(x):
    return 1 / (1 +np.exp(-x))

In [ ]:
FEATURE_MODEL_PATH = '/workspace/shared_data/models/feature_model'
MODEL_PATH = '/workspace/shared_data/models/model'
 
feature_model = tf.keras.models.load_model(FEATURE_MODEL_PATH, custom_objects=custom_objects)
model = tf.keras.models.load_model(MODEL_PATH, custom_objects=custom_objects)

# 5. For Pre-processing (defined in our paper)
- Adjust the parameter and do test.

In [ ]:
test_seq = iaa.Sequential([
    iaa.GaussianBlur(sigma=(0, 0.7)),
    iaa.Dropout((0.01, 0.15), per_channel=0.5),
    iaa.Affine(
        scale={"x": (0.9, 1.1), "y": (0.9, 1.1)},
        translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
        rotate=(-30, 30),
        order=[0, 1],
        cval=1
    )
], random_order=True)

# 6. Split the 1,200 test set to 3 part (For three clusters)

## 6-1. Extract the feature vectors

In [ ]:
# AUTH only needs the final FC weight/bias to project the query feature vector.
# (Enrollment of result1/2/3 already happened in the enroll notebook -> ctxt1.)
weight = model.weights[30]
bias = model.weights[31]
print('weight shape:', weight.shape, '| bias shape:', bias.shape)

## 6-2 Encrypt the three sets of the feature vectors 

## TEST

In [ ]:
from time import time
import os
MAX_FINGER_NUM = 512
THRESHOLD = 0.99 # Set your threshold
URL = 'http://main:8090/blindtouch' # main server, reachable by container name on the docker network

# Query an ENROLLED fingerprint so the HE matching is actually exercised.
# ctxt1 enrolled test_data[:512]; with a single cluster the meaningful scores are
# the i=0 block -> result_list[0:512], where result_list[k] is the match score for
# enrolled print k. Override with BT_AUTH_IDX. (idx>=512 would be an un-enrolled
# print and should always Reject.)
idx = int(os.environ.get('BT_AUTH_IDX', str(random.randint(0, 511))))
print('query idx :', idx)
test = test_seq.augment_image(test_data[idx]) # Pre-processing

START_TIME = time()
scaled_x_test = []

predicted_ = feature_model.predict(test.reshape([1,224,224,1]))
predicted_ = keras.layers.UnitNormalization()(layers.Flatten()(predicted_))
predicted = np.matmul(predicted_, weight) + bias
predicted = np.array(predicted.numpy().tolist()[0]* MAX_FINGER_NUM)


ctxt = encryptor.encrypt(ckks_encoder.encode(predicted, scale))
ctxt.save('temp_ctxt')
files = open('temp_ctxt', 'rb')
upload = {'target_enc': files}
response = requests.post(URL, files = upload)

with open('./result_enc', mode='wb') as localfile:
    localfile.write(response.content)
FINISH_TIME = time()
ctxt2 = Ciphertext()
ctxt2.load(context, './result_enc')

vec = ckks_encoder.decode(decryptor.decrypt(ctxt2))
print('TOTAL TIME (s): ', time()-START_TIME)
result_list = []
for i in range(3):
    for j in range(512):
        result_list.append(sigmoid(vec[j * 16 + i]))

# Single cluster -> meaningful scores live in the i=0 block (first 512 entries).
cluster0 = result_list[:512]
matches = [ii for ii in range(len(result_list)) if result_list[ii] > THRESHOLD]
top = sorted(range(512), key=lambda k: cluster0[k], reverse=True)[:5]
print('self-match score (enrolled idx %d): %.6f' % (idx, cluster0[idx]))
print('max score over enrolled set      : %.6f (idx %d)' % (max(cluster0), int(np.argmax(cluster0))))
print('top-5 enrolled scores            :', [(t, round(cluster0[t], 6)) for t in top])
print('indices over THRESHOLD %.2f      : %s' % (THRESHOLD, matches))
print('====================================')
print('RESULT:', 'Authenticated' if matches else 'Rejected')
print('====================================')